# Multi-Agent AI Academic Assistant
## Jupyter Notebook — Submission Demo

This notebook demonstrates the core components of the Multi-Agent AI Academic Assistant:
1. **Dataset exploration** — course catalogue files (raw and cleaned/chunked)
2. **RAG pipeline** — document loading, chunking, embedding, and retrieval
3. **Agent interactions** — Tutor, Analytics, Support, and Orchestrator agents
4. **LLM usage** — OpenAI GPT-4o-mini via LangChain and AutoGen
5. **Sample outputs** — real agent responses

## 0. Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path("../").resolve()
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
print("✅ Environment loaded")
print(f"OpenAI key set: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No — add to .env file!'}")

## 1. Dataset Exploration — Raw Course Catalogue

In [ ]:
import pandas as pd
from pypdf import PdfReader

catalogue_dir = project_root / "data" / "course_catalogue"
files = sorted([f for f in catalogue_dir.iterdir() if f.suffix.lower() in {".pdf", ".txt", ".docx"}])

print("=== RAW DATASET: Course Catalogue ===")
print(f"Number of course files: {len(files)}
")

file_stats = []
for f in files:
    if f.suffix.lower() == ".pdf":
        reader = PdfReader(str(f))
        num_pages = len(reader.pages)
        # Extract text from first page for preview
        preview = reader.pages[0].extract_text()[:150].strip() if reader.pages else "(empty)"
        file_stats.append({"Course": f.stem, "Pages": num_pages, "Type": "PDF"})
        print(f"📄 {f.stem}")
        print(f"   Pages: {num_pages} | Type: PDF")
        print(f"   Preview: {preview}...
")
    else:
        text = f.read_text(encoding="utf-8")
        words = len(text.split())
        file_stats.append({"Course": f.stem, "Pages": f"{words:,} words", "Type": f.suffix.upper()})
        print(f"📄 {f.stem}")
        print(f"   Words: {words:,} | Type: {f.suffix.upper()}")
        print(f"   Preview: {text[:150].strip()}...
")

df_files = pd.DataFrame(file_stats)
print("
=== Dataset Summary ===")
print(df_files.to_string(index=False))

## 2. Data Cleaning Pipeline — Chunking and Preprocessing

In [ ]:
from rag.document_loader import load_directory, load_file, CHUNK_SIZE, CHUNK_OVERLAP

print("=== CLEANING PIPELINE CONFIGURATION ===")
print(f"Chunk size: {CHUNK_SIZE} characters")
print(f"Chunk overlap: {CHUNK_OVERLAP} characters")
print(f"Separators: paragraph breaks → line breaks → sentences → words")
print()

# Load and chunk all catalogue files
print("=== LOADING AND CHUNKING DOCUMENTS ===")
all_docs = load_directory(str(catalogue_dir))
print(f"Total chunks produced: {len(all_docs)}")
print(f"Average chunk size: {sum(len(d.page_content) for d in all_docs) // len(all_docs)} chars")
print()

# Show example chunks
print("=== SAMPLE CLEANED CHUNKS ===")
for i, doc in enumerate(all_docs[:3]):
    print(f"\n--- Chunk {i+1} (source: {doc.metadata.get('source', 'unknown')}) ---")
    print(doc.page_content[:300])
    print(f"[...{len(doc.page_content)} chars total]")

In [ ]:
# Visualize chunk distribution
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter

source_counts = Counter(doc.metadata.get("source", "unknown") for doc in all_docs)
chunk_lengths = [len(doc.page_content) for doc in all_docs]

# Bar chart of chunks per course
fig1 = go.Figure(go.Bar(
    x=list(source_counts.keys()),
    y=list(source_counts.values()),
    marker_color='steelblue',
    text=list(source_counts.values()),
    textposition='outside'
))
fig1.update_layout(title="Chunks per Course (Cleaned Dataset)",
                   xaxis_title="Course", yaxis_title="Number of Chunks")
fig1.show()

# Histogram of chunk lengths
fig2 = px.histogram(x=chunk_lengths, nbins=20, title="Chunk Length Distribution",
                    labels={"x": "Chunk Length (chars)", "y": "Count"})
fig2.show()

print(f"Min chunk: {min(chunk_lengths)} | Max: {max(chunk_lengths)} | Mean: {sum(chunk_lengths)//len(chunk_lengths)}")

## 3. Vector Store — Embeddings (Cleaned Dataset)

In [ ]:
from rag.vector_store import build_catalogue_store, similarity_search

print("Building vector store (ChromaDB)...")
print("Model: text-embedding-3-small (OpenAI)")
print("Storage: persistent ChromaDB at data/chroma_db/")
print()

store = build_catalogue_store(all_docs)
print(f"✅ Vector store built with {len(all_docs)} embedded chunks")

# Test retrieval
query = "What is backpropagation in neural networks?"
results = similarity_search(store, query, k=3)

print(f"\n=== RETRIEVAL TEST ===")
print(f"Query: '{query}'")
print(f"Top {len(results)} results:\n")
for i, doc in enumerate(results):
    print(f"Result {i+1} (source: {doc.metadata.get('source')})")
    print(doc.page_content[:200])
    print()

## 4. Agent Demonstrations

In [ ]:
from utils.session import StudentSession
from agents.tutor import TutorAgent
from rag.vector_store import get_retriever

# --- TUTOR AGENT ---
print("=" * 60)
print("TUTOR AGENT — RAG-powered Q&A")
print("=" * 60)

tutor = TutorAgent()
retriever = get_retriever(store, k=4)
tutor.set_retriever(retriever)

questions = [
    "Explain gradient descent and its variants.",
    "What is the difference between overfitting and underfitting?",
]

for q in questions:
    print(f"\n📚 Question: {q}")
    result = tutor.answer(q)
    print(f"\n🤖 TutorAgent:")
    print(result['answer'][:600])
    print(f"\n📎 Sources: {result['sources']}")
    print("-" * 50)

In [ ]:
# --- QUIZ GENERATION ---
print("=" * 60)
print("TUTOR AGENT — Quiz Generation")
print("=" * 60)

quiz = tutor.generate_quiz("Binary Trees", num_questions=3)
print(quiz)

In [ ]:
from agents.analytics import AnalyticsAgent

# Build a sample session with data
session = StudentSession("Demo Student")
session.add_topic("Machine Learning")
session.add_topic("Neural Networks")
session.add_topic("Python Basics")
session.add_quiz_result("Machine Learning", 8, 10)
session.add_quiz_result("Neural Networks", 5, 10)
session.add_quiz_result("Python Basics", 9, 10)
session.add_quiz_result("Sorting Algorithms", 4, 10)

# --- ANALYTICS AGENT ---
print("=" * 60)
print("ANALYTICS AGENT — Performance Analysis")
print("=" * 60)

analytics = AnalyticsAgent()
result = analytics.analyze(session)
print(result['analysis'])

In [ ]:
# Visualize analytics data
data = analytics.get_progress_data(session)

fig = go.Figure(go.Bar(
    x=data['quiz_topics'],
    y=data['quiz_scores'],
    marker_color=['#2ecc71' if s >= 80 else '#f39c12' if s >= 60 else '#e74c3c'
                  for s in data['quiz_scores']],
    text=[f"{s}%" for s in data['quiz_scores']],
    textposition='outside'
))
fig.update_layout(
    title=f"Student Performance Dashboard — {session.student_name}",
    yaxis_title="Score (%)",
    yaxis_range=[0, 115],
    xaxis_title="Topic",
    plot_bgcolor='rgba(0,0,0,0)'
)
fig.show()
print(f"Average score: {data['average_score']}% | Weak areas: {data['weak_areas']}")

In [ ]:
from agents.support import SupportAgent

# --- SUPPORT AGENT ---
print("=" * 60)
print("SUPPORT AGENT — Student Wellbeing")
print("=" * 60)

support = SupportAgent()
result = support.respond(
    "I'm feeling overwhelmed with exams coming up and I can't focus on studying."
)
print(result['answer'])

In [ ]:
from agents.orchestrator import OrchestratorAgent

# --- ORCHESTRATOR — intent routing ---
print("=" * 60)
print("ORCHESTRATOR AGENT — Intelligent Routing")
print("=" * 60)

orch = OrchestratorAgent(session=session, openai_api_key=os.getenv("OPENAI_API_KEY"))
orch.set_retriever(retriever)

test_queries = [
    ("Explain what a binary search tree is.", "→ Expected: TUTOR"),
    ("How am I doing in my studies?", "→ Expected: ANALYTICS"),
    ("I'm stressed about my upcoming exam.", "→ Expected: SUPPORT"),
]

for query, expected in test_queries:
    intent = orch._classify_intent(query)
    print(f"Query: '{query}'")
    print(f"{expected} | Actual: {intent}\n")

## 5. Multi-Agent Collaboration (AutoGen)

In [ ]:
# Demonstrate the AutoGen collaboration workflow
print("=" * 60)
print("MULTI-AGENT COLLABORATION (AutoGen)")
print("TutorBot + AnalyticsBot working together")
print("=" * 60)
print()

collab_query = "I keep failing neural network quizzes. Can you help me understand them better?"
print(f"Complex query: '{collab_query}'")
print("Routing to COLLABORATION mode (AutoGen group chat)...\n")

result = orch._run_collaboration(collab_query)
print("=== COLLABORATIVE RESPONSE ===")
print(result['answer'])

## 6. Dataset Summary

| Property | Value |
|---|---|
| **Raw Dataset** | 4 OpenStax PDF textbooks (Intro to CS, Intro Statistics 2e, Principles of Data Science, Python Programming) |
| **Total pages** | ~3,000+ across all 4 PDFs |
| **License** | CC BY 4.0 (OpenStax / Rice University) |
| **Cleaning steps** | PyPDF extraction → RecursiveCharacterTextSplitter (800 char chunks, 100 overlap) |
| **Cleaned dataset** | ChromaDB collection with thousands of embedded chunks |
| **Embedding model** | OpenAI text-embedding-3-small |
| **Assumptions** | PDFs loaded page by page; chunk overlap preserves context across page/section boundaries |
| **Student uploads** | PDF/DOCX/TXT files dynamically embedded and added to ChromaDB per session |

### Cleaning Steps Explained
1. Load raw PDF files using  (LangChain)
2. Split with  — tries paragraph breaks first, then line breaks, then sentences, then words
3. Each chunk retains metadata:  (filename) and 
4. Chunks are embedded with OpenAI  and stored in ChromaDB at 
5. At query time, top-k most similar chunks are retrieved and injected into the LLM prompt as context